In [4]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import transformers
from huggingface_hub import list_repo_files
from transformers import AutoConfig

In [5]:
import transformers

print(transformers.__version__)

4.41.2


In [6]:
files = list_repo_files("soketlabs/pragna-1b")

for f in files:
    print(f)

.gitattributes
README.md
config.json
generation_config.json
model.safetensors
pragna_hf.png
special_tokens_map.json
tokenizer.json
tokenizer_config.json


In [7]:
config = AutoConfig.from_pretrained("soketlabs/pragna-1b")

print(config)

LlamaConfig {
  "_name_or_path": "soketlabs/pragna-1b",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.41.2",
  "use_cache": true,
  "vocab_size": 69632
}



In [9]:
weights_path = hf_hub_download(
    "soketlabs/pragna-1b",
    filename="model.safetensors"
)

state_dict = load_file(weights_path)

output_file = "pragna_tensor_inventory.txt"

with open(output_file, "w", encoding="utf-8") as f:

    f.write(f"Number of tensors: {len(state_dict)}\n\n")

    for key, tensor in state_dict.items():

        line = f"{key:<80} {tuple(tensor.shape)}\n"

        f.write(line)

print(f"Saved tensor inventory to {output_file}")

Saved tensor inventory to pragna_tensor_inventory.txt


In [10]:
output_file = "pragna_layer0.txt"

with open(output_file, "w", encoding="utf-8") as f:

    for key, tensor in state_dict.items():

        if key.startswith("model.layers.0"):

            f.write(
                f"{key:<80} {tuple(tensor.shape)}\n"
            )

print(f"Saved to {output_file}")

Saved to pragna_layer0.txt


In [11]:
output_file = "pragna_summary.txt"

with open(output_file, "w", encoding="utf-8") as f:

    f.write("=== EMBEDDINGS ===\n")

    for k, v in state_dict.items():

        if "embed_tokens" in k or "lm_head" in k:
            f.write(f"{k} {tuple(v.shape)}\n")

    f.write("\n=== LAYER 0 ===\n")

    for k, v in state_dict.items():

        if k.startswith("model.layers.0"):
            f.write(f"{k} {tuple(v.shape)}\n")

    f.write("\n=== FINAL NORM ===\n")

    for k, v in state_dict.items():

        if "model.norm" in k:
            f.write(f"{k} {tuple(v.shape)}\n")

print("Summary saved.")

Summary saved.
